In [1]:
import numpy as np
import pandas as pd
import joblib
import os

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED, N_MELS

# Entrenamiento de Random Forest

Entrenamos un RF para cada tipo de procesamiento aplicado a ciclos/ventanas.

También se probaron otros ensembles como ``GradientBoostingClassifier`` o ``HistGradientBoostingClassifier``, sin embargo, el rendimiento fue similar.

Elegimos un dataset de ciclos respiratorios o ventanas temporales

In [2]:
dataset = 'ciclos/mic'

## Espectrogramas Mel

Cargamos las matrices guardadas en el path correspondiente

In [12]:
train_data = np.load(f'./data_procesada/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./data_procesada/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [13]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((3336, 47616), (3336,), (835, 47616), (835,))

### Eleccion de hiperparámetros

Idealmente usamos `GridSearchCV` y/o `RandomizedSearchCV` para probar distintas convinaciones de hiperparametros sin sobreajustar a los datos de entrenamiento.

Nuestra métrica principal a medir es el área bajo la curva ROC.

El único hiperparámetro fijo es ``max_features='sqrt'``, tenemos demasiados features y sería computacionalmente costoso que el modelo considere todos en cada paso. Además, `max_features='log2` tuvo un peor desempeño que 'sqrt'. 

In [17]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')

param_distributions = {
    'max_depth': [10, 20, 30],
    'min_samples_split': [20, 40, 60],
    'min_samples_leaf': [10, 20, 30],
}

In [18]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.695 total time=   3.6s
[CV 2/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.761 total time=   3.8s
[CV 3/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.710 total time=   6.2s
[CV 4/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.700 total time=   5.2s
[CV 5/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.698 total time=   5.5s
[CV 1/5] END max_depth=20, min_samples_leaf=20, min_samples_split=40;, score=0.714 total time=   6.7s


KeyboardInterrupt: 

Vemos los resultados de la validación cruzada

In [7]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,20,10,20,0.719193
1,20,20,40,0.714039
8,20,20,20,0.714039
3,30,20,20,0.713939
5,20,10,60,0.711439
9,30,30,20,0.705773
6,20,30,40,0.705636
7,20,30,60,0.705636
4,10,10,20,0.703212
0,10,30,60,0.697213


Elegimos y, de ser necesario, entrenamos el modelo. Si elegimos el mejor modelo del CV lo podemos escoger ya entrenado.

In [ ]:
best_model = rnd_search.best_estimator_

In [14]:
best_model = RandomForestClassifier(
    max_depth=14,
    min_samples_split=26,
    min_samples_leaf=12,
    max_features='sqrt',
    random_state=SEED,
    n_jobs=-1,
)

best_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,14
,min_samples_split,26
,min_samples_leaf,12
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### Evaluación rápida

In [15]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.45      0.56       359
           1       0.68      0.88      0.77       476

    accuracy                           0.70       835
   macro avg       0.71      0.67      0.67       835
weighted avg       0.71      0.70      0.68       835



Y lo guardamos en la carpeta correspondiente

In [16]:
os.makedirs(f'./modelos_clasicos/modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/melspec_rf.pkl')

['./modelos_clasicos/modelos/ciclos/mic/melspec_rf.pkl']

## Atributos de Audio

Cargamos los features guardados en la carpeta del dataset elegido

In [3]:
train_data = np.load(f'./data_procesada/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./data_procesada/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((3336, 46), (3336,), (835, 46), (835,))

### Eleccion de hiperparámetros

Igual que antes, usamos `GridSearchCV` y/o `RandomizedSearchCV` para probar distintas convinaciones de hiperparametros sin sobreajustar a los datos de entrenamiento.

Nuestra métrica principal a medir es el área bajo la curva ROC.

El único hiperparámetro fijo es `max_features=None` ya que en este caso no tenemos una gran cantidad de features y se probó que funciona mejor ``'sqrt'`` y ``'log2'``

In [5]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features=None)

param_distributions = {
    'max_depth': [10, 20, 30],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10, 15]
}

In [6]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.821 total time=   1.1s
[CV 2/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.810 total time=   1.0s
[CV 3/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.808 total time=   1.1s
[CV 4/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.821 total time=   1.1s
[CV 5/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.816 total time=   1.2s
[CV 1/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.838 total time=   1.7s
[CV 2/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.832 total time=   1.8s
[CV 3/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.838 total time=   2.1s
[CV 4/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.843 total time=   2.0s
[CV 5/5] END max_dept

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20, ...], 'min_samples_leaf': [5, 10, ...], 'min_samples_split': [10, 20, ...]}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


Observamos los resultados

In [7]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,20,5,10,0.850894
1,20,10,20,0.837609
8,20,10,10,0.837609
3,30,10,10,0.837600
5,20,5,30,0.834916
4,10,5,10,0.833122
7,20,15,30,0.824115
6,20,15,20,0.824115
9,30,15,10,0.824115
0,10,15,30,0.815253


Nos quedamos con un modelo

In [8]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': 20}
Best CV score: 0.8508942226409602


### Evaluación rápida

In [9]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.71      0.77       359
           1       0.80      0.89      0.85       476

    accuracy                           0.82       835
   macro avg       0.82      0.80      0.81       835
weighted avg       0.82      0.82      0.81       835



Y guardamos el modelo

In [11]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_rf.pkl')

['./modelos_clasicos/modelos/ciclos/mic/features_rf.pkl']